# Module 1, Notebook 6: The Python Ecosystem

## For Senior Swift Engineers

This notebook covers the tools and conventions that make up a professional Python development environment. As a Swift engineer, you already understand the importance of package management (SPM), testing (XCTest), linting, and project structure. Python has its own ecosystem for all of these, and understanding it is essential for writing production-quality code.

**Note:** Since this is a Jupyter notebook, we will demonstrate concepts with code cells where possible. Some tools (like `pytest`, `ruff`, `mypy`) are command-line tools that you would run in your terminal. We will show the commands and explain the output.

---

## Swift vs Python Quick Reference

| Concept | Swift | Python |
|---|---|---|
| Package manager | Swift Package Manager (SPM) | pip, pip-tools, uv |
| Package manifest | `Package.swift` | `pyproject.toml` |
| Dependency lock file | `Package.resolved` | `requirements.txt` / `uv.lock` |
| Isolated environments | N/A (Xcode manages) | `venv` (virtual environments) |
| Test framework | XCTest | pytest |
| Test discovery | XCTest auto-discovers `test*` | pytest auto-discovers `test_*` |
| Mocking | Protocol-based / dependency injection | `unittest.mock` |
| Linter | SwiftLint | ruff (replaces flake8, isort, etc.) |
| Formatter | swift-format | ruff format (replaces black) |
| Type checker | Built into compiler | mypy (separate tool) |
| Debugging | Xcode debugger / `lldb` | `pdb` / `breakpoint()` / IDE debugger |
| Entry point | `@main` / `main.swift` | `__main__.py` / `if __name__ == '__main__'` |
| Module init | N/A (implicit) | `__init__.py` |

---

## 1. Virtual Environments

### Why they exist (no Swift equivalent needed)

In Swift/Xcode, each project has its own dependencies managed by SPM -- there is no global package pollution problem. Python is different: by default, `pip install` installs packages **globally** (or per-user). This creates problems:

- Project A needs `requests==2.28` but Project B needs `requests==2.31`
- Installing packages globally can break system tools
- You cannot reproduce someone else's environment reliably

**Virtual environments** solve this by creating isolated Python installations per project.

### How it works

```bash
# Create a virtual environment (like creating a fresh Xcode project)
python3 -m venv .venv

# Activate it (tells your shell to use this Python)
source .venv/bin/activate   # macOS/Linux
# .venv\Scripts\activate    # Windows

# Now pip install goes into .venv/, not globally
pip install requests

# See what's installed in this environment
pip list

# Deactivate when done
deactivate
```

### What `.venv/` contains

```
.venv/
  bin/           # Python executable + scripts (activate, pip, etc.)
  lib/           # Installed packages (site-packages)
  include/       # C headers (for packages with C extensions)
  pyvenv.cfg     # Configuration file
```

**Always add `.venv/` to your `.gitignore`** -- it is a local artifact, not source code.

In [ ]:
# You can check if you're in a virtual environment from Python:

import sys

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"Virtual env: {sys.prefix != sys.base_prefix}")  # True if in a venv

# In a real project, you would see something like:
# Python executable: /path/to/project/.venv/bin/python
# Virtual env: True

---

## 2. Package Management

### pip (the default)

```bash
# Install a package
pip install requests

# Install a specific version
pip install requests==2.31.0

# Install from requirements file
pip install -r requirements.txt

# Freeze current environment (creates requirements.txt)
pip freeze > requirements.txt

# Uninstall
pip uninstall requests
```

### pip-tools (better dependency management)

`pip-tools` separates direct dependencies from transitive ones (like SPM's `Package.swift` vs `Package.resolved`):

```bash
pip install pip-tools

# requirements.in — your direct dependencies (like Package.swift)
# requirements.txt — locked/resolved dependencies (like Package.resolved)

# Compile: resolve dependencies and create locked file
pip-compile requirements.in -o requirements.txt

# Sync: install exactly what's in requirements.txt
pip-sync requirements.txt
```

### uv (the modern, fast option)

`uv` is a Rust-based replacement for pip, pip-tools, and virtualenv. It is **10-100x faster** than pip.

```bash
# Install uv
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create a project (like swift package init)
uv init my-project
cd my-project

# Add dependencies (like adding to Package.swift)
uv add requests
uv add pytest --dev  # development dependency

# Run a command in the project's environment
uv run python main.py
uv run pytest

# Lock dependencies
uv lock

# Sync environment to match lock file
uv sync
```

**Recommendation for 2025+:** Use `uv` for new projects. It handles virtual environments, dependency resolution, and locking in one tool.

---

## 3. `pyproject.toml` -- The Modern Package Manifest

Think of `pyproject.toml` as Python's `Package.swift`. It is the single file that defines your project's metadata, dependencies, build system, and tool configuration.

### Swift comparison

```swift
// Package.swift
let package = Package(
    name: "MyApp",
    platforms: [.macOS(.v13)],
    dependencies: [
        .package(url: "https://github.com/Alamofire/Alamofire.git", from: "5.0.0"),
    ],
    targets: [
        .target(name: "MyApp", dependencies: ["Alamofire"]),
        .testTarget(name: "MyAppTests", dependencies: ["MyApp"]),
    ]
)
```

In [ ]:
# Here is what a typical pyproject.toml looks like:

pyproject_example = '''
[project]
name = "my-ml-app"
version = "0.1.0"
description = "A machine learning application"
readme = "README.md"
requires-python = ">= 3.11"
authors = [
    { name = "Daniel", email = "daniel@example.com" },
]

# Direct dependencies (like Package.swift dependencies)
dependencies = [
    "numpy>=1.24",
    "pandas>=2.0",
    "scikit-learn>=1.3",
    "requests>=2.31",
]

# Optional dependency groups (like SPM's conditional dependencies)
[project.optional-dependencies]
dev = [
    "pytest>=7.0",
    "pytest-cov>=4.0",
    "ruff>=0.4",
    "mypy>=1.8",
]

# Entry points (like @main in Swift)
[project.scripts]
my-ml-app = "my_ml_app.cli:main"

# Build system configuration
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

# === Tool configuration (all in one file!) ===

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"

[tool.ruff]
line-length = 100
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "I", "N", "UP"]

[tool.mypy]
python_version = "3.11"
strict = true
'''

print(pyproject_example)

---

## 4. Testing with pytest

pytest is Python's most popular testing framework. It is simpler and more powerful than `unittest` (Python's built-in, which is modeled after Java's JUnit).

### Swift comparison

```swift
// XCTest
import XCTest

class CalculatorTests: XCTestCase {
    func testAddition() {
        XCTAssertEqual(Calculator.add(2, 3), 5)
    }
    
    func testDivisionByZero() {
        XCTAssertThrowsError(try Calculator.divide(1, 0))
    }
}
```

Key differences from XCTest:
- **No test class required** -- just write functions starting with `test_`
- **Use plain `assert`** -- no `XCTAssertEqual`, `XCTAssertTrue`, etc.
- **Fixtures** replace `setUp`/`tearDown`
- **Parametrize** lets you run the same test with different data

In [ ]:
# Let's create a module to test

class Calculator:
    """A simple calculator for demonstration."""
    
    @staticmethod
    def add(a: float, b: float) -> float:
        return a + b
    
    @staticmethod
    def subtract(a: float, b: float) -> float:
        return a - b
    
    @staticmethod
    def multiply(a: float, b: float) -> float:
        return a * b
    
    @staticmethod
    def divide(a: float, b: float) -> float:
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

print("Calculator module defined.")

In [ ]:
# What a pytest test file looks like: tests/test_calculator.py
# Note: In a notebook we can't actually run pytest, but this shows the syntax.

test_file_content = '''
# tests/test_calculator.py
import pytest
from my_app.calculator import Calculator


# --- Basic tests (just functions, no class needed!) ---

def test_add():
    """Test basic addition."""
    assert Calculator.add(2, 3) == 5        # XCTAssertEqual equivalent
    assert Calculator.add(-1, 1) == 0
    assert Calculator.add(0, 0) == 0


def test_subtract():
    assert Calculator.subtract(5, 3) == 2
    assert Calculator.subtract(3, 5) == -2


def test_divide():
    assert Calculator.divide(10, 2) == 5.0
    assert Calculator.divide(7, 2) == 3.5


def test_divide_by_zero():
    """Test that dividing by zero raises ValueError."""
    # Like XCTAssertThrowsError
    with pytest.raises(ValueError, match="Cannot divide by zero"):
        Calculator.divide(1, 0)


# --- Parametrize: run same test with multiple inputs ---
# No XCTest equivalent! (You would write separate test methods or loop.)

@pytest.mark.parametrize("a, b, expected", [
    (1, 1, 2),
    (0, 0, 0),
    (-1, 1, 0),
    (100, 200, 300),
    (0.1, 0.2, pytest.approx(0.3)),  # floating point comparison!
])
def test_add_parametrized(a, b, expected):
    assert Calculator.add(a, b) == expected


# --- Group related tests in a class (optional, like XCTestCase) ---

class TestMultiply:
    """Group multiplication tests together."""
    
    def test_positive_numbers(self):
        assert Calculator.multiply(3, 4) == 12
    
    def test_negative_numbers(self):
        assert Calculator.multiply(-3, 4) == -12
        assert Calculator.multiply(-3, -4) == 12
    
    def test_multiply_by_zero(self):
        assert Calculator.multiply(5, 0) == 0
'''

print(test_file_content)

In [ ]:
# Fixtures — pytest's replacement for setUp/tearDown
# Fixtures provide reusable test dependencies via dependency injection.

fixture_example = '''
# tests/conftest.py — shared fixtures (auto-discovered by pytest)
import pytest
import tempfile
from pathlib import Path


# --- Fixture basics ---

@pytest.fixture
def sample_data():
    """Provide sample test data."""
    return {
        "users": [
            {"name": "Alice", "age": 30},
            {"name": "Bob", "age": 25},
        ]
    }


# Use it by adding the fixture name as a parameter — pytest injects it!
def test_user_count(sample_data):  # <-- pytest sees 'sample_data' and calls the fixture
    assert len(sample_data["users"]) == 2


# --- Fixture with setup AND teardown (like setUp/tearDown in XCTest) ---

@pytest.fixture
def temp_dir():
    """Create a temporary directory for the test, clean up after."""
    dir_path = Path(tempfile.mkdtemp())
    yield dir_path         # <-- yield = test runs here
    # Teardown: everything after yield runs after the test
    import shutil
    shutil.rmtree(dir_path)


def test_file_creation(temp_dir):
    """Test that uses a temporary directory."""
    test_file = temp_dir / "test.txt"
    test_file.write_text("hello")
    assert test_file.read_text() == "hello"
    # temp_dir is automatically cleaned up after this test!


# --- Fixture scopes (like XCTestCase class-level setup) ---

@pytest.fixture(scope="module")  # shared across all tests in a module
def db_connection():
    """Create a database connection once per test module."""
    conn = create_connection()  # expensive setup
    yield conn
    conn.close()  # teardown

# Scopes: "function" (default), "class", "module", "session"
'''

print(fixture_example)

In [ ]:
# Markers — tagging and filtering tests

marker_example = '''
import pytest

# Mark a test as slow (skip during quick test runs)
@pytest.mark.slow
def test_large_dataset_processing():
    """This test takes a while."""
    # ... process 1M records ...
    pass

# Skip a test unconditionally
@pytest.mark.skip(reason="Not implemented yet")
def test_future_feature():
    pass

# Skip based on a condition
@pytest.mark.skipif(
    sys.platform != "darwin",
    reason="macOS only"
)
def test_macos_specific():
    pass

# Mark a test as expected to fail (like XCTExpectFailure)
@pytest.mark.xfail(reason="Known bug #123")
def test_known_bug():
    assert buggy_function() == "expected"

# Run specific markers:
# pytest -m "not slow"      # skip slow tests
# pytest -m "slow"           # only slow tests
# pytest -k "test_add"       # only tests matching pattern
'''

print(marker_example)

In [ ]:
# Mocking with unittest.mock
# In Swift, you typically use protocol-based DI for testability.
# Python's mock is more dynamic — it can patch any attribute at runtime.

from unittest.mock import Mock, patch, MagicMock

# --- Mock basics ---

# Create a mock object that records all calls
mock_api = Mock()
mock_api.get_user.return_value = {"name": "Alice", "age": 30}

# Use the mock
result = mock_api.get_user(42)
print(f"Result: {result}")

# Verify the mock was called correctly (like XCTest expectations)
mock_api.get_user.assert_called_once_with(42)
print(f"Call count: {mock_api.get_user.call_count}")
print(f"Call args: {mock_api.get_user.call_args}")

In [ ]:
# @patch — temporarily replace a module-level object during a test
# This is Python's superpower for testing — no protocol abstraction needed!

from unittest.mock import patch
import json

# A function that makes a "real" API call (we want to mock this in tests)
def fetch_weather(city: str) -> dict:
    """Fetch weather data from an API."""
    import urllib.request
    url = f"https://api.weather.example.com/{city}"
    response = urllib.request.urlopen(url)  # This would make a real HTTP call
    return json.loads(response.read())

# In a test, we patch urllib.request.urlopen to avoid real HTTP calls:
mock_response_data = json.dumps({"temp": 72, "condition": "sunny"}).encode()

with patch('urllib.request.urlopen') as mock_urlopen:
    # Configure the mock
    mock_urlopen.return_value.__enter__ = Mock(return_value=Mock())
    mock_urlopen.return_value.read.return_value = mock_response_data
    
    # Now fetch_weather won't make a real HTTP call
    # (In a real test, this would work; in the notebook the patching
    #  targets differently, so let's demonstrate the concept)
    print("Mock configured successfully")
    print(f"mock_urlopen called: {mock_urlopen.called}")

# The decorator form (used in actual test files):
test_with_mock = '''
@patch('my_app.weather.urllib.request.urlopen')
def test_fetch_weather(mock_urlopen):
    mock_urlopen.return_value.read.return_value = b'{"temp": 72}'
    result = fetch_weather("SF")
    assert result["temp"] == 72
    mock_urlopen.assert_called_once()
'''
print(test_with_mock)

In [ ]:
# Running pytest — common commands

pytest_commands = """
# Run all tests
pytest

# Verbose output (like XCTest's detailed reporting)
pytest -v

# Run specific test file
pytest tests/test_calculator.py

# Run specific test function
pytest tests/test_calculator.py::test_add

# Run tests matching a pattern
pytest -k "add or multiply"

# Stop on first failure
pytest -x

# Show print output (normally captured)
pytest -s

# Run with coverage report
pytest --cov=my_app --cov-report=term-missing

# Run only failed tests from last run
pytest --lf

# Run tests in parallel (with pytest-xdist)
pytest -n auto
"""

print(pytest_commands)

---

## 5. Code Quality Tools

### ruff — The All-in-One Linter + Formatter

ruff replaces multiple tools (flake8, isort, pyflakes, pycodestyle, black) with a single, blazing-fast Rust-based tool. Think of it as SwiftLint + swift-format combined.

```bash
# Install
pip install ruff

# Lint (find problems)
ruff check .                 # like: swiftlint lint
ruff check --fix .           # auto-fix what it can

# Format (like swift-format or black)
ruff format .                # format all files
ruff format --check .        # check without changing
```

In [ ]:
# ruff configuration in pyproject.toml

ruff_config = '''
[tool.ruff]
# Target Python version
target-version = "py311"

# Max line length (default 88, like Black)
line-length = 100

# Exclude directories
exclude = [
    ".venv",
    "__pycache__",
    "migrations",
]

[tool.ruff.lint]
# Select rule categories:
# E = pycodestyle errors
# F = pyflakes (unused imports, undefined names)
# I = isort (import sorting)
# N = pep8-naming (naming conventions)
# UP = pyupgrade (use modern Python syntax)
# B = flake8-bugbear (common bugs)
# SIM = flake8-simplify (simplify code)
# RUF = ruff-specific rules
select = ["E", "F", "I", "N", "UP", "B", "SIM", "RUF"]

# Ignore specific rules
ignore = [
    "E501",  # line too long (let formatter handle it)
]

[tool.ruff.lint.isort]
# Import sorting (like SwiftLint's sorted_imports rule)
known-first-party = ["my_app"]

[tool.ruff.format]
# Formatting options
quote-style = "double"   # use double quotes
indent-style = "space"   # spaces, not tabs
'''

print(ruff_config)

In [ ]:
# mypy — static type checking
# This is the closest Python gets to Swift's compile-time type checking.

mypy_examples = '''
# Example code that mypy would check:

def greet(name: str) -> str:
    return f"Hello, {name}"

# mypy would catch this:
greet(42)  # error: Argument 1 has incompatible type "int"; expected "str"

# mypy catches None-safety issues (like Swift optionals):
from typing import Optional

def find_user(user_id: int) -> Optional[str]:
    users = {1: "Alice"}
    return users.get(user_id)  # Returns str | None

name = find_user(1)
print(name.upper())  # error: Item "None" has no attribute "upper"

# Fixed:
if name is not None:     # Swift equivalent: if let name = find_user(1)
    print(name.upper())  # OK — mypy knows name is str here
'''

# mypy configuration in pyproject.toml
mypy_config = '''
[tool.mypy]
python_version = "3.11"
strict = true                # Enable all strict checks
warn_return_any = true
warn_unused_configs = true

# Per-module overrides (like per-file SwiftLint rules)
[[tool.mypy.overrides]]
module = "tests.*"
disallow_untyped_defs = false  # tests can be less strict
'''

print("--- mypy examples ---")
print(mypy_examples)
print("--- mypy config ---")
print(mypy_config)

---

## 6. Project Structure Conventions

Python has strong conventions about how to organize a project. Understanding these is like understanding the Xcode project structure -- it is how the ecosystem expects things to be laid out.

In [ ]:
# Standard Python project layout

project_structure = """
my-project/                     # Repository root
|
|-- pyproject.toml              # Package manifest (like Package.swift)
|-- README.md
|-- LICENSE
|-- .gitignore
|
|-- src/                        # Source root (recommended "src layout")
|   |-- my_app/                 # Your package (note: underscores, not hyphens)
|   |   |-- __init__.py         # Makes this directory a Python package
|   |   |-- __main__.py         # Entry point: python -m my_app
|   |   |-- cli.py              # CLI interface
|   |   |-- config.py           # Configuration handling
|   |   |-- models/             # Sub-package
|   |   |   |-- __init__.py
|   |   |   |-- user.py
|   |   |   |-- product.py
|   |   |-- services/
|   |   |   |-- __init__.py
|   |   |   |-- auth.py
|   |   |   |-- database.py
|   |   |-- utils/
|   |       |-- __init__.py
|   |       |-- helpers.py
|
|-- tests/                      # Test directory (separate from source)
|   |-- __init__.py
|   |-- conftest.py             # Shared pytest fixtures
|   |-- test_cli.py
|   |-- test_config.py
|   |-- models/
|   |   |-- test_user.py
|   |   |-- test_product.py
|   |-- services/
|       |-- test_auth.py
|       |-- test_database.py
|
|-- docs/                       # Documentation
|-- scripts/                    # Helper scripts
"""

print(project_structure)

In [ ]:
# Understanding key special files

print("=== __init__.py ===")
print("""
Purpose: Makes a directory into a Python package (module).
Swift equivalent: No direct equivalent — Swift modules are defined by targets.

Can be empty, or can contain:
- Package-level imports for convenience
- __all__ to control 'from package import *'
- Package initialization code

Example __init__.py:
  # src/my_app/__init__.py
  __version__ = "0.1.0"
  
  # Re-export for convenience (like Swift's @_exported import)
  from .models.user import User
  from .models.product import Product
  
  __all__ = ["User", "Product"]
""")

print("=== __main__.py ===")
print("""
Purpose: Entry point when running a package as a script.
Swift equivalent: @main struct or main.swift

  # src/my_app/__main__.py
  from .cli import main
  main()

Run with: python -m my_app
""")

print("=== if __name__ == '__main__' ===")
print("""
Purpose: Code that runs only when the file is executed directly.
Does NOT run when the file is imported as a module.

  def main():
      print("Running!")
  
  if __name__ == '__main__':
      main()

  # python my_script.py       -> prints "Running!"
  # import my_script           -> does NOT print
""")

---

## 7. Debugging

### Swift comparison

In Swift/Xcode, you set breakpoints in the IDE and use the debugger GUI. Python has both command-line and IDE-based debugging.

In [ ]:
# breakpoint() — Python 3.7+ built-in
# This is the easiest way to drop into a debugger.
# In a notebook, it won't work interactively, but in a script:

debug_example = '''
def process_data(items):
    results = []
    for item in items:
        # Drop into debugger here (like setting a breakpoint in Xcode)
        breakpoint()  # opens pdb (Python DeBugger)
        
        result = transform(item)
        results.append(result)
    return results

# When breakpoint() hits, you get a pdb prompt:
# (Pdb) p item          # print a variable (like 'po' in lldb)
# (Pdb) n               # next line (step over)
# (Pdb) s               # step into function
# (Pdb) c               # continue execution
# (Pdb) l               # list source code around current line
# (Pdb) pp results      # pretty-print a variable
# (Pdb) w               # show call stack (like 'bt' in lldb)
# (Pdb) q               # quit debugger
'''

print(debug_example)

# Tip: Install 'ipdb' for a better debugging experience:
# pip install ipdb
# Then set: PYTHONBREAKPOINT=ipdb.set_trace
# Now breakpoint() uses ipdb (with syntax highlighting, tab completion)

In [ ]:
# Other debugging techniques

# 1. Rich tracebacks with 'rich' library
# pip install rich
# from rich import traceback; traceback.install()
# Now all tracebacks are beautifully formatted

# 2. Quick debugging with print() / repr()
data = {"key": [1, 2, 3], "nested": {"a": True}}
print(f"Debug: {data!r}")  # !r forces repr() — shows types clearly

# 3. f-string debugging (Python 3.8+) — the = sign
x = 42
name = "Daniel"
items = [1, 2, 3]
print(f"{x = }")        # x = 42
print(f"{name = }")     # name = 'Daniel'
print(f"{len(items) = }")  # len(items) = 3
# This is incredibly useful for quick debugging!

---

## 8. Common Standard Library Gems

Python's standard library is famously "batteries included." Here are modules you will use constantly in ML/AI work.

In [ ]:
# collections — specialized container datatypes

from collections import Counter, defaultdict, deque, namedtuple

# Counter — count occurrences (like NSCountedSet)
words = "the quick brown fox jumps over the lazy dog the fox".split()
word_counts = Counter(words)
print(f"Word counts: {word_counts}")
print(f"Most common: {word_counts.most_common(3)}")

# defaultdict — dict with default values for missing keys
groups = defaultdict(list)  # missing keys get an empty list
for name, dept in [("Alice", "Eng"), ("Bob", "Sales"), ("Charlie", "Eng")]:
    groups[dept].append(name)  # no KeyError!
print(f"\nGroups: {dict(groups)}")

# deque — double-ended queue (O(1) append/pop from both ends)
d = deque([1, 2, 3], maxlen=5)  # with optional max length
d.appendleft(0)
d.append(4)
print(f"\nDeque: {d}")
print(f"Popleft: {d.popleft()}")  # O(1), unlike list.pop(0) which is O(n)

# namedtuple — lightweight immutable data class
Point = namedtuple('Point', ['x', 'y'])
p = Point(3, 4)
print(f"\nPoint: {p}, x={p.x}, y={p.y}")

In [ ]:
# functools — higher-order functions and operations on callables

from functools import partial, reduce, lru_cache

# partial — pre-fill some arguments of a function
# Swift equivalent: not built-in (you'd write a closure)
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)
print(f"square(5) = {square(5)}")
print(f"cube(3) = {cube(3)}")

# reduce — fold a sequence to a single value
# Swift equivalent: .reduce(0, +)
numbers = [1, 2, 3, 4, 5]
total = reduce(lambda acc, x: acc + x, numbers, 0)
print(f"\nreduce sum: {total}")

# lru_cache — we covered this in Notebook 4, but it's worth revisiting
@lru_cache(maxsize=256)
def expensive_lookup(key: str) -> str:
    """Simulate an expensive operation."""
    print(f"  Computing for {key}...")
    return f"result_{key}"

print(f"\n{expensive_lookup('a')}")  # computes
print(f"{expensive_lookup('a')}")    # cached!
print(f"{expensive_lookup('b')}")    # computes

In [ ]:
# itertools — iterator building blocks (covered more in Notebook 4)

import itertools

# accumulate — running totals (like Swift's scan)
running_sum = list(itertools.accumulate([1, 2, 3, 4, 5]))
print(f"Running sum: {running_sum}")

# batched (Python 3.12+) — split into chunks
try:
    batches = list(itertools.batched(range(10), 3))
    print(f"Batched: {batches}")
except AttributeError:
    # For older Python versions:
    def batched(iterable, n):
        it = iter(iterable)
        while True:
            batch = list(itertools.islice(it, n))
            if not batch:
                break
            yield tuple(batch)
    print(f"Batched: {list(batched(range(10), 3))}")

# pairwise (Python 3.10+) — sliding window of size 2
try:
    pairs = list(itertools.pairwise([1, 2, 3, 4, 5]))
    print(f"Pairwise: {pairs}")
except AttributeError:
    print("pairwise requires Python 3.10+")

# combinations and permutations
print(f"Combos C(4,2): {list(itertools.combinations('ABCD', 2))}")
print(f"Perms P(3,2): {list(itertools.permutations('ABC', 2))}")

In [ ]:
# dataclasses — already covered in Notebook 3, but essential to mention here

from dataclasses import dataclass, field
from typing import ClassVar

@dataclass
class Config:
    """Application configuration."""
    host: str = "localhost"
    port: int = 8080
    debug: bool = False
    tags: list[str] = field(default_factory=list)
    
    # Class variable (not included in __init__)
    max_connections: ClassVar[int] = 100

@dataclass(frozen=True)  # immutable, like Swift struct
class Point:
    x: float
    y: float
    
    @property
    def magnitude(self) -> float:
        return (self.x ** 2 + self.y ** 2) ** 0.5

config = Config(host="0.0.0.0", debug=True, tags=["production"])
print(f"Config: {config}")

p = Point(3.0, 4.0)
print(f"Point: {p}, magnitude: {p.magnitude}")

# Frozen dataclass is immutable:
try:
    p.x = 5.0
except AttributeError as e:
    print(f"Cannot modify frozen: {e}")

In [ ]:
# Other useful standard library modules

# --- textwrap — text formatting ---
import textwrap

long_text = "This is a very long string that needs to be wrapped at a reasonable width for display in a terminal or other fixed-width output."
print(textwrap.fill(long_text, width=50))
print()

# --- decimal — precise decimal arithmetic ---
from decimal import Decimal

# Floating point issues:
print(f"float: 0.1 + 0.2 = {0.1 + 0.2}")
# Decimal is precise:
print(f"Decimal: 0.1 + 0.2 = {Decimal('0.1') + Decimal('0.2')}")
print()

# --- secrets — cryptographically strong random numbers ---
import secrets

token = secrets.token_hex(16)  # 32-character hex string
print(f"Secret token: {token}")
api_key = secrets.token_urlsafe(32)
print(f"API key: {api_key}")
print()

# --- enum — enumerations (like Swift enums, but simpler) ---
from enum import Enum, auto

class Color(Enum):
    RED = auto()
    GREEN = auto()
    BLUE = auto()

print(f"Color: {Color.RED}, name={Color.RED.name}, value={Color.RED.value}")

---

## Exercises

### Exercise 1: Set Up a Project with `pyproject.toml`

Create a complete `pyproject.toml` for a fictional "data pipeline" project that:
1. Has proper metadata (name, version, description, author)
2. Requires Python 3.11+
3. Has dependencies: `pandas`, `requests`, `pydantic`
4. Has dev dependencies: `pytest`, `pytest-cov`, `ruff`, `mypy`
5. Defines a CLI entry point
6. Configures pytest, ruff, and mypy
7. Also sketch out the directory structure

In [ ]:
# Exercise 1: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 1: Solution

pyproject_solution = '''
[project]
name = "data-pipeline"
version = "0.1.0"
description = "A data pipeline for processing and transforming datasets"
readme = "README.md"
license = { text = "MIT" }
requires-python = ">= 3.11"
authors = [
    { name = "Daniel", email = "daniel@example.com" },
]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3.11",
    "Programming Language :: Python :: 3.12",
]

dependencies = [
    "pandas>=2.0",
    "requests>=2.31",
    "pydantic>=2.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.0",
    "pytest-cov>=4.0",
    "pytest-mock>=3.10",
    "ruff>=0.4",
    "mypy>=1.8",
    "pandas-stubs",         # type stubs for pandas
    "types-requests",       # type stubs for requests
]

[project.scripts]
data-pipeline = "data_pipeline.cli:main"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

# === Tool Configuration ===

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = [
    "-v",
    "--tb=short",
    "--strict-markers",
]
markers = [
    "slow: marks tests as slow (deselect with '-m not slow')",
    "integration: marks integration tests",
]

[tool.ruff]
target-version = "py311"
line-length = 100
exclude = [".venv", "__pycache__", ".mypy_cache"]

[tool.ruff.lint]
select = [
    "E",    # pycodestyle errors
    "F",    # pyflakes
    "I",    # isort
    "N",    # pep8-naming
    "UP",   # pyupgrade
    "B",    # flake8-bugbear
    "SIM",  # flake8-simplify
    "RUF",  # ruff-specific
]

[tool.ruff.lint.isort]
known-first-party = ["data_pipeline"]

[tool.mypy]
python_version = "3.11"
strict = true
warn_return_any = true
warn_unused_configs = true
plugins = ["pydantic.mypy"]

[[tool.mypy.overrides]]
module = "tests.*"
disallow_untyped_defs = false
'''

print(pyproject_solution)

print("\n" + "=" * 60)
print("Directory structure:")
print("=" * 60)

structure = """
data-pipeline/
|-- pyproject.toml
|-- README.md
|-- LICENSE
|-- .gitignore              # include: .venv/, __pycache__/, *.egg-info/, dist/
|
|-- src/
|   |-- data_pipeline/
|   |   |-- __init__.py     # __version__ = "0.1.0"
|   |   |-- __main__.py     # from .cli import main; main()
|   |   |-- cli.py          # Click/argparse CLI
|   |   |-- config.py       # Pydantic settings
|   |   |-- models.py       # Pydantic data models
|   |   |-- pipeline.py     # Core pipeline logic
|   |   |-- sources/
|   |   |   |-- __init__.py
|   |   |   |-- csv_source.py
|   |   |   |-- api_source.py
|   |   |-- transforms/
|   |   |   |-- __init__.py
|   |   |   |-- clean.py
|   |   |   |-- aggregate.py
|   |   |-- sinks/
|   |       |-- __init__.py
|   |       |-- csv_sink.py
|   |       |-- database_sink.py
|
|-- tests/
|   |-- __init__.py
|   |-- conftest.py         # Shared fixtures
|   |-- test_pipeline.py
|   |-- test_models.py
|   |-- sources/
|   |   |-- test_csv_source.py
|   |   |-- test_api_source.py
|   |-- transforms/
|       |-- test_clean.py
|       |-- test_aggregate.py
"""
print(structure)

### Exercise 2: Write Tests for a Module

Write comprehensive pytest tests for the `UserService` class below. Include:
1. Basic tests for CRUD operations
2. Parametrized tests for validation
3. Tests that verify exceptions are raised
4. A fixture for setting up test data
5. A mock for the external `email_service`

In [ ]:
# The module to test

from dataclasses import dataclass, field
import re

@dataclass
class User:
    id: int
    name: str
    email: str

class ValidationError(Exception):
    pass

class NotFoundError(Exception):
    pass

class DuplicateError(Exception):
    pass

class UserService:
    """Service for managing users."""
    
    EMAIL_REGEX = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')
    
    def __init__(self, email_service=None):
        self._users: dict[int, User] = {}
        self._next_id = 1
        self._email_service = email_service
    
    def _validate_email(self, email: str) -> None:
        if not self.EMAIL_REGEX.match(email):
            raise ValidationError(f"Invalid email: {email}")
    
    def _validate_name(self, name: str) -> None:
        if not name or len(name.strip()) < 2:
            raise ValidationError(f"Name must be at least 2 characters")
    
    def create_user(self, name: str, email: str) -> User:
        self._validate_name(name)
        self._validate_email(email)
        
        # Check for duplicate email
        for user in self._users.values():
            if user.email == email:
                raise DuplicateError(f"Email already exists: {email}")
        
        user = User(id=self._next_id, name=name.strip(), email=email.lower())
        self._users[user.id] = user
        self._next_id += 1
        
        # Send welcome email
        if self._email_service:
            self._email_service.send_welcome(user.email, user.name)
        
        return user
    
    def get_user(self, user_id: int) -> User:
        if user_id not in self._users:
            raise NotFoundError(f"User {user_id} not found")
        return self._users[user_id]
    
    def list_users(self) -> list[User]:
        return list(self._users.values())
    
    def delete_user(self, user_id: int) -> None:
        if user_id not in self._users:
            raise NotFoundError(f"User {user_id} not found")
        del self._users[user_id]

print("UserService defined. Write tests below!")

In [ ]:
# Exercise 2: Your solution here
# Write the tests as if in a test file, then we'll run some assertions directly.

# YOUR CODE HERE
pass

In [ ]:
# Exercise 2: Solution
# These tests follow pytest conventions. We will run them directly here,
# but in a real project they would be in tests/test_user_service.py.

from unittest.mock import Mock, call

# --- Fixture equivalent (in a real test file, use @pytest.fixture) ---

def make_service(with_email=False):
    """Fixture: create a UserService with optional mock email service."""
    email_mock = Mock() if with_email else None
    service = UserService(email_service=email_mock)
    return service, email_mock

def make_populated_service():
    """Fixture: service with pre-populated test data."""
    service, _ = make_service()
    service.create_user("Alice", "alice@example.com")
    service.create_user("Bob", "bob@example.com")
    return service

# --- Basic CRUD tests ---

def test_create_user():
    service, _ = make_service()
    user = service.create_user("Alice", "alice@example.com")
    assert user.id == 1
    assert user.name == "Alice"
    assert user.email == "alice@example.com"

def test_create_user_strips_name():
    service, _ = make_service()
    user = service.create_user("  Alice  ", "alice@example.com")
    assert user.name == "Alice"

def test_create_user_lowercases_email():
    service, _ = make_service()
    user = service.create_user("Alice", "Alice@EXAMPLE.COM")
    assert user.email == "alice@example.com"

def test_get_user():
    service = make_populated_service()
    user = service.get_user(1)
    assert user.name == "Alice"

def test_list_users():
    service = make_populated_service()
    users = service.list_users()
    assert len(users) == 2
    assert {u.name for u in users} == {"Alice", "Bob"}

def test_delete_user():
    service = make_populated_service()
    service.delete_user(1)
    assert len(service.list_users()) == 1

def test_auto_increment_ids():
    service, _ = make_service()
    u1 = service.create_user("Alice", "a@example.com")
    u2 = service.create_user("Bob", "b@example.com")
    assert u1.id == 1
    assert u2.id == 2

# --- Validation tests (parametrized in real pytest) ---

invalid_emails = ["", "not-an-email", "@missing.com", "missing@", "no-dot@com"]
invalid_names = ["", " ", "A", "  x  "]

def test_invalid_emails():
    """Parametrized: test various invalid emails."""
    for email in invalid_emails:
        service, _ = make_service()
        try:
            service.create_user("Valid Name", email)
            assert False, f"Should have raised ValidationError for: {email!r}"
        except ValidationError:
            pass  # expected

def test_invalid_names():
    """Parametrized: test various invalid names."""
    for name in invalid_names:
        service, _ = make_service()
        try:
            service.create_user(name, "valid@example.com")
            assert False, f"Should have raised ValidationError for name: {name!r}"
        except ValidationError:
            pass  # expected

# --- Exception tests ---

def test_get_nonexistent_user():
    service, _ = make_service()
    try:
        service.get_user(999)
        assert False, "Should have raised NotFoundError"
    except NotFoundError as e:
        assert "999" in str(e)

def test_delete_nonexistent_user():
    service, _ = make_service()
    try:
        service.delete_user(999)
        assert False, "Should have raised NotFoundError"
    except NotFoundError:
        pass

def test_duplicate_email():
    service, _ = make_service()
    service.create_user("Alice", "alice@example.com")
    try:
        service.create_user("Bob", "alice@example.com")
        assert False, "Should have raised DuplicateError"
    except DuplicateError:
        pass

# --- Mock tests ---

def test_welcome_email_sent():
    service, email_mock = make_service(with_email=True)
    service.create_user("Alice", "alice@example.com")
    email_mock.send_welcome.assert_called_once_with("alice@example.com", "Alice")

def test_no_email_without_service():
    service, _ = make_service(with_email=False)
    # Should not raise even without email service
    service.create_user("Alice", "alice@example.com")

# Run all tests!
test_functions = [
    test_create_user, test_create_user_strips_name, test_create_user_lowercases_email,
    test_get_user, test_list_users, test_delete_user, test_auto_increment_ids,
    test_invalid_emails, test_invalid_names,
    test_get_nonexistent_user, test_delete_nonexistent_user, test_duplicate_email,
    test_welcome_email_sent, test_no_email_without_service,
]

passed = 0
failed = 0
for test in test_functions:
    try:
        test()
        passed += 1
        print(f"  PASSED: {test.__name__}")
    except Exception as e:
        failed += 1
        print(f"  FAILED: {test.__name__} — {e}")

print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")

### Exercise 3: Configure Ruff for a Project

Write a complete ruff configuration in `pyproject.toml` format that:
1. Targets Python 3.11
2. Uses a line length of 100
3. Selects comprehensive rule sets
4. Ignores specific rules with comments explaining why
5. Configures import sorting with first-party package
6. Sets up per-file ignores for test files

Then show examples of code that would be flagged by ruff and the fixed versions.

In [ ]:
# Exercise 3: Your solution here

# YOUR CODE HERE
pass

In [ ]:
# Exercise 3: Solution

ruff_solution = '''
# === pyproject.toml — Ruff Configuration ===

[tool.ruff]
# Target Python version
target-version = "py311"

# Max line length
line-length = 100

# Directories to exclude
exclude = [
    ".venv",
    "__pycache__",
    ".mypy_cache",
    ".pytest_cache",
    "dist",
    "build",
    "migrations",  # auto-generated database migrations
]

[tool.ruff.lint]
# Rule categories to enable:
select = [
    # Core rules
    "E",     # pycodestyle errors (whitespace, indentation)
    "W",     # pycodestyle warnings
    "F",     # pyflakes (unused imports, undefined names)
    
    # Import organization
    "I",     # isort (import sorting)
    
    # Code quality
    "N",     # pep8-naming (naming conventions)
    "UP",    # pyupgrade (use modern Python syntax)
    "B",     # flake8-bugbear (common bugs)
    "SIM",   # flake8-simplify (simplify code)
    "C4",    # flake8-comprehensions (better comprehensions)
    "DTZ",   # flake8-datetimez (timezone-aware datetime)
    "RUF",   # ruff-specific rules
    
    # Documentation
    "D",     # pydocstyle (docstring conventions)
    
    # Type checking
    "TCH",   # flake8-type-checking (TYPE_CHECKING imports)
    "ANN",   # flake8-annotations (type annotation issues)
]

# Rules to ignore (with justification)
ignore = [
    "E501",   # Line too long — let the formatter handle wrapping
    "D100",   # Missing docstring in public module — too strict for now
    "D104",   # Missing docstring in public package — __init__.py doesn't need one
    "ANN101", # Missing type annotation for self — obvious from context
    "ANN102", # Missing type annotation for cls — obvious from context
]

# Per-file rule overrides
[tool.ruff.lint.per-file-ignores]
# Tests can be less strict:
"tests/**/*.py" = [
    "D",       # No docstring requirements for tests
    "ANN",     # No type annotation requirements for tests
    "S101",    # Allow assert in tests (it's the whole point!)
]
# Scripts don't need full documentation:
"scripts/**/*.py" = ["D", "ANN"]

[tool.ruff.lint.isort]
known-first-party = ["data_pipeline"]
# Group imports: stdlib, third-party, first-party, local
section-order = ["future", "standard-library", "third-party", "first-party", "local-folder"]

[tool.ruff.lint.pydocstyle]
convention = "google"  # Use Google-style docstrings

[tool.ruff.format]
quote-style = "double"
indent-style = "space"
skip-magic-trailing-comma = false
line-ending = "auto"
'''

print(ruff_solution)

In [ ]:
# Examples of code ruff would flag and the fixed versions

examples = """
# === Example 1: Import sorting (I001) ===

# BEFORE (ruff would flag):
import os
import requests
import sys
from data_pipeline import models
from pathlib import Path
import json

# AFTER (ruff fix):
import json
import os
import sys
from pathlib import Path

import requests

from data_pipeline import models


# === Example 2: Modern Python syntax (UP) ===

# BEFORE:
from typing import List, Dict, Optional, Tuple
def process(items: List[Dict[str, int]]) -> Optional[Tuple[str, ...]]: ...

# AFTER (Python 3.10+ syntax):
def process(items: list[dict[str, int]]) -> tuple[str, ...] | None: ...


# === Example 3: Bugbear (B006) — mutable default argument ===

# BEFORE (dangerous — shared mutable default!):
def add_item(item, items=[]):
    items.append(item)
    return items

# AFTER:
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items


# === Example 4: Simplify (SIM) ===

# BEFORE:
if x == True:    # SIM: use 'if x:'
    pass
if len(items) > 0:  # SIM: use 'if items:'
    pass
    
# AFTER:
if x:
    pass
if items:
    pass


# === Example 5: Comprehensions (C4) ===

# BEFORE:
result = dict([(k, v) for k, v in items.items() if v > 0])

# AFTER:
result = {k: v for k, v in items.items() if v > 0}
"""

print(examples)

---

## Key Takeaways

1. **Virtual environments** are essential. Always create one per project (`python -m venv .venv`). This is something Swift/Xcode handles automatically, but Python requires explicit management.

2. **`pyproject.toml`** is your `Package.swift`. It defines everything: metadata, dependencies, tool configuration. Use it as the single source of truth.

3. **uv** is the recommended modern tool for package management. It is fast, handles venvs + deps + locking in one tool, and is rapidly becoming the standard.

4. **pytest** is far more concise than XCTest. Plain `assert` statements, fixtures for DI, `@parametrize` for data-driven tests, and `unittest.mock` for patching anything.

5. **ruff** replaces multiple tools (flake8, isort, black) with one fast binary. Configure it in `pyproject.toml` and run it in CI.

6. **mypy** gives you the closest experience to Swift's compile-time type safety. Use `strict = true` in new projects.

7. **Project structure matters**: Use the `src/` layout, put `__init__.py` in every package directory, and keep tests separate from source code.

8. **The standard library is huge**: `collections`, `functools`, `itertools`, `pathlib`, and `dataclasses` will cover many needs without third-party packages.

---

*This concludes Module 1: Python Foundations for Swift Engineers. You now have a solid grounding in Python's syntax, data structures, OOP, advanced features, file I/O, error handling, and the development ecosystem. Next up: Module 2, where we dive into the data science and ML stack.*